
<h1 id="Differential-photometry">Differential photometry</h1><p>Through this Jupyter notebook, we will see how to use the <em>aperture photometry</em> of our target and reference star(s) to obtain the <em>differential photometry</em> of our target.</p>
<p>To understand the difference between <em>aperture photometry</em> and <em>differential photometry</em>, let's first analyze the output of our <em>AperturePhotometry</em> class.</p>
<p>First, we load one of the science frames to highlight the selected stars, the parameters of the sky annulus, and the aperture of choice, following the results obtained during the previous lectures.</p>


In [ ]:


import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget
from astropy.io import fits 
import pickle 
import matplotlib.colors as colors




In [ ]:


science_frame_name = './group00_GJ3470_20220328/correct/AF771843_corr.p'
science_corrected =    pickle.load(open(science_frame_name, 'rb'))





<p>The two following functions have been taken from the previous lectures.</p>


In [ ]:


def make_annulus_around_star(x_pos, y_pos,  inner_radius, outer_radius, label='', color='y'):
    from matplotlib.patches import Circle

    n, radii = 50, [inner_radius, outer_radius]
    theta = np.linspace(0, 2*np.pi, n, endpoint=True)
    xs = np.outer(radii, np.cos(theta))
    ys = np.outer(radii, np.sin(theta))
    
    # in order to have a closed area, the circles
    # should be traversed in opposite directions
    xs[1,:] = xs[1,::-1]
    ys[1,:] = ys[1,::-1]
    
    ax.fill(np.ravel(xs)+x_pos, np.ravel(ys)+y_pos, edgecolor=None, facecolor=color, alpha=0.75, label=label)



def make_circle_around_star(x_pos, y_pos, radius, thickness=0.5, label='', color='w', alpha=1.):
    from matplotlib.patches import Circle

    n, radii = 50, [radius, radius+thickness]
    theta = np.linspace(0, 2*np.pi, n, endpoint=True)
    xs = np.outer(radii, np.cos(theta))
    ys = np.outer(radii, np.sin(theta))
    
    # in order to have a closed area, the circles
    # should be traversed in opposite directions
    xs[1,:] = xs[1,::-1]
    ys[1,:] = ys[1,::-1]
    
    ax.fill(np.ravel(xs)+x_pos, np.ravel(ys)+y_pos, edgecolor=None, facecolor=color, alpha=alpha, label=label)





<p>The coordinates of the target and reference stars and the aperture photometry parameters come from the analysis completed in the earlier lectures. \</p>
<p>You can use the following plot to refine the choice of the coordinates and the inner/ outer radius for all the stars.<br/>
This is how it looks like with a poor choice of coordinates:</p>


In [ ]:


x_target = 350
y_target = 26

x_reference_01 = 175
y_reference_01 = 46

x_reference_02 = 70
y_reference_02 = 28


aperture = 8 # 6 10
inner_radius = 15
outer_radius = 23

vmin = np.amin(science_corrected[:,100:400])
vmax = np.amax(science_corrected[:,100:400])
print('vmin:  {0:.1f}    vmax: {1:.1f}'.format(vmin, vmax))
vmax = 5000

fig, ax = plt.subplots(1, figsize=(8,2))
im1 = plt.imshow(science_corrected, cmap=plt.colormaps['magma'], norm=colors.LogNorm(vmin=vmin, vmax=vmax), origin='lower')
plt.colorbar(im1,ax=ax, fraction=0.046, pad=0.04)

make_circle_around_star(x_target, y_target, aperture, label='Aperture')
make_circle_around_star(x_reference_01, y_reference_01, aperture)
make_circle_around_star(x_reference_02, y_reference_02, aperture)

make_annulus_around_star(x_target, y_target, inner_radius, outer_radius, label='Target')
make_annulus_around_star(x_reference_01, y_reference_01, inner_radius, outer_radius, label='Reference #1', color='g')
make_annulus_around_star(x_reference_02, y_reference_02, inner_radius, outer_radius, label='Reference #2', color='r')

plt.xlabel(' X [pixels]')
plt.ylabel(' Y [pixels]')
ax.legend()
plt.show()





<p><strong>The coordinates do not have to be perfect</strong>, the centroid algorithm will take care of it, but you want a good starting point. <br/>
This is the same plot, but after manually adjusting the coordinates of the targets:</p>


In [ ]:


x_target = 345
y_target = 33

x_reference_01 = 169
y_reference_01 = 37

x_reference_02 = 60
y_reference_02 = 28

aperture = 8 # 6 10
inner_radius = 15
outer_radius = 23

vmin = np.amin(science_corrected[:,100:400])
vmax = np.amax(science_corrected[:,100:400])
print('vmin:  {0:.1f}    vmax: {1:.1f}'.format(vmin, vmax))
vmax = 5000

fig, ax = plt.subplots(1, figsize=(8,2))
im1 = plt.imshow(science_corrected, cmap=plt.colormaps['magma'], norm=colors.LogNorm(vmin=vmin, vmax=vmax), origin='lower')
plt.colorbar(im1,ax=ax, fraction=0.046, pad=0.04)

make_circle_around_star(x_target, y_target, aperture, label='Aperture')
make_circle_around_star(x_reference_01, y_reference_01, aperture)
make_circle_around_star(x_reference_02, y_reference_02, aperture)

make_annulus_around_star(x_target, y_target, inner_radius, outer_radius, label='Target')
make_annulus_around_star(x_reference_01, y_reference_01, inner_radius, outer_radius, label='Reference #1', color='g')
make_annulus_around_star(x_reference_02, y_reference_02, inner_radius, outer_radius, label='Reference #3', color='r')

plt.xlabel(' X [pixels]')
plt.ylabel(' Y [pixels]')
ax.legend()
plt.show()





<h2 id="Importing-the-AperturePhotometry-class">Importing the <em>AperturePhotometry</em> class</h2><p>The best way to use a class is to save it in an external <em>.py</em> file to import it effortlessly and safely in several Jupyter notebooks, for example, if you want to test different aperture values or reference stars.</p>
<p>In this example, I save the class <code>AperturePhotometry</code> inside the file <code>aperture_class.py</code>. The standard approach to load the class is to use the line:</p>
<div class="highlight"><pre><span></span><span class="kn">from</span><span class="w"> </span><span class="nn">aperture_class</span><span class="w"> </span><span class="kn">import</span> <span class="n">AperturePhotometry</span>
</pre></div>
<p>Notice how the file name is called without extension.
The drawback of this approach - and classes in general - is that you will only notice errors, or <em>bugs</em>, when initializing the class or using one of its methods.
Regardless of your experience, writing such a complex class without <em>any</em> mistakes is nearly impossible, and even a typo may prevent your class from working. Inevitably, you will need to do some <em>debugging</em>.
To fix any errors, you must edit the file where your class is saved and reload it again.  However, python is following an <a href="https://stackoverflow.com/questions/65655755/why-does-python-import-only-once"><em>import-only-once rule</em></a>, so rerunning the above command will not import the updated class. Consequently, you will keep getting the same error even if it is now fixed in the script file, as you will still be working with the class imported early on before fixing the problem.
There are two workarounds for this problem:</p>
<ul>
<li>reload the kernel, but you need to run all the cells again</li>
<li>at least during the debugging phase, import the class in the following way to ensure that the most updated version of the class is imported:</li>
</ul>
<div class="highlight"><pre><span></span><span class="kn">from</span><span class="w"> </span><span class="nn">importlib</span><span class="w"> </span><span class="kn">import</span> <span class="n">reload</span>     <span class="c1"># you need to run this line only once </span>
<span class="kn">import</span><span class="w"> </span><span class="nn">aperture_class</span>            <span class="c1"># you need to run this line only once </span>

<span class="c1"># for the next line to work, aperture_class must be imported once already </span>
<span class="n">reload</span><span class="p">(</span><span class="n">aperture_class</span><span class="p">)</span>           
<span class="c1"># we finally get the most updated version of AperturePhotometry</span>
<span class="kn">from</span><span class="w"> </span><span class="nn">aperture_class</span><span class="w"> </span><span class="kn">import</span> <span class="n">AperturePhotometry</span> 
</pre></div>


In [ ]:
from aperture_class import AperturePhotometry

ModuleNotFoundError: No module named 'aperture_class'


<h2 id="Extracting-the-aperture-photometry">Extracting the aperture photometry</h2><p>We now use the <code>AperturePhotometry</code> class to perform aperture photometry on the target and two reference stars. For your analysis, do not select more than three reference stars among the brightest in the field of view, other than the target star. <br/>
Initialization, attributes, and methods may change depending on how you code your <code>AperturePhotometry</code> class.</p>


In [ ]:


from time import time
t0 = time()

aperture = 8
inner_radius = 18
outer_radius = 23

target_ap08 = AperturePhotometry()
target_ap08.provide_aperture_parameters(inner_radius, outer_radius, aperture, x_target, y_target)
target_ap08.aperture_photometry()

reference01_ap08 = AperturePhotometry()
reference01_ap08.provide_aperture_parameters(inner_radius, outer_radius, aperture, x_reference_01, y_reference_01)
reference01_ap08.aperture_photometry()

reference02_ap08 = AperturePhotometry()
reference02_ap08.provide_aperture_parameters(inner_radius, outer_radius, aperture, x_reference_02, y_reference_02)
reference02_ap08.aperture_photometry()

t1 = time()
print('elapsed_time=', t1-t0)





<p>Remember that the time of observations must be converted from Julian date in <a href="https://en.wikipedia.org/wiki/Coordinated_Universal_Time">Coordinated Universal Time (UTC) standard</a>, measured at the beginning of the exposure, into the <a href="https://ui.adsabs.harvard.edu/abs/2010PASP..122..935E/abstract">Barycentric Julian Date in the Barycentric Dynamical Time (BJD_TDB) standard</a> at mid exposure.</p>
<p>You can perform this step either inside the class or as a separate step.
From now on, we will use the BJD_TDB of the target star for all the plots, as the reference stars have been observed with the same time stamps.</p>



<h2 id="Tracking-weather-and-instrument-conditions-during-the-night">Tracking weather and instrument conditions during the night</h2><p>The plot below shows the aperture photometry of our stars and other observables of interest as a function of time.</p>
<ol>
<li>in the first panel, the <strong>flux</strong> of each star is reported. Each flux has been normalized to the value of the <code>normalization_index</code>-th observation to better compare the flux of stars with different magnitudes (and hence different absolute flux)</li>
<li>in the second panel, the <strong>airmass</strong> as a function of time is shown.</li>
<li>in the third panel, the <strong>sky flux</strong> measured around each star is reported. Higher values of the sky background may be associated with increasing airmass (light pollution due to nearby villages) but also with variable sky weather conditions, e.g., an increase in cloud coverage. This panel can be helpful if you want to exclude some observations (e.g., those affected by extremely high values of sky background, like near the end of the night in my case) or to exclude a reference star if the associated sky background has a strange behaviour compared to the other stars (likely due to the presence of a contaminant star near the sky annulus).</li>
<li>the fourth panel shows the <strong>variation of the position of the target</strong> on the CCD during the night in both horizontal and vertical directions. In this example, the variation is relative to the position measured in the first scientific frame, but any frame can be chosen. You may want to exclude observations affected by sudden jumps</li>
<li>The last panel shows the <strong>FWHM</strong> of the target, in both directions. You will have this information only if you implemented the optional FWHM determination correctly in the <code>AperturePhotometry</code> class.</li>
</ol>
<p>Mind the several tricks to improve the plot's readability.</p>
<ul>
<li><code>time_offset</code> is subtracted to each time stamp to make the horizontal scale more readable.</li>
<li>You can change the vertical range using <code>ax.set_ylim</code></li>
<li>You can use the logarithmic scale for some axis</li>
</ul>


In [ ]:


normalization_index = 200
time_offset = round(np.median(target_ap08.bjd_tdb))

fig, axs = plt.subplots(5, 1, sharex=True, figsize=(8,10))
# Reduce vertical space between axes
fig.subplots_adjust(hspace=0.05)

axs[0].scatter(target_ap08.bjd_tdb-time_offset, target_ap08.aperture/target_ap08.aperture[normalization_index], s=2, zorder=3, c='C0', label='Target')
axs[0].scatter(target_ap08.bjd_tdb-time_offset, reference01_ap08.aperture/reference01_ap08.aperture[normalization_index], s=2, zorder=2, c='C1', label='Ref #1')
axs[0].scatter(target_ap08.bjd_tdb-time_offset, reference02_ap08.aperture/reference02_ap08.aperture[normalization_index], s=2, zorder=1, c='C2', label='Ref #2')
#axs[0].set_yticks(np.arange(0.90, 1.1, 0.025))
#axs[0].set_ylim(0.88, 1.052)
axs[0].set_ylabel('Normalized flux')
axs[0].legend()

axs[1].scatter(target_ap08.bjd_tdb-time_offset, target_ap08.airmass, s=2, c='C0', label='Airmass')
axs[1].set_ylabel('Airmass')

axs[2].scatter(target_ap08.bjd_tdb-time_offset, target_ap08.sky_background, s=2, zorder=3, c='C0', label='Target')
axs[2].scatter(target_ap08.bjd_tdb-time_offset, reference01_ap08.sky_background, s=2, zorder=2, c='C1', label='Ref #1')
axs[2].scatter(target_ap08.bjd_tdb-time_offset, reference02_ap08.sky_background, s=2, zorder=1, c='C2', label='Ref #2')
axs[2].set_ylabel('Sky background [photons]')
#axs[2].set_yscale('log')
axs[2].legend()

axs[3].scatter(target_ap08.bjd_tdb-time_offset, target_ap08.x_refined-target_ap08.x_refined[0], s=2, zorder=3, c='C0', label='X direction')
axs[3].scatter(target_ap08.bjd_tdb-time_offset, target_ap08.y_refined-target_ap08.y_refined[0], s=2, zorder=2, c='C1', label='Y direction')
axs[3].set_ylabel('Telescope drift [pixels]')
axs[3].legend()

axs[4].scatter(target_ap08.bjd_tdb-time_offset, target_ap08.x_fwhm, s=2, zorder=3, c='C0', label='X direction')
axs[4].scatter(target_ap08.bjd_tdb-time_offset, target_ap08.y_fwhm, s=2, zorder=2, c='C1', label='Y direction')
axs[4].set_ylabel('Target fWHM [pixels]')
axs[4].legend()


axs[4].set_xlabel('BJD-TDB - {0:.1f} [days]'.format(time_offset))
plt.show()





<h2 id="Computing-the-differential-photometry">Computing the differential photometry</h2><p>Focusing on the target star in the first panel shows that the absolute flux is highly variable during the night. You may have acknowledged that these variations can be seen due to changes in the atmospheric seeing, weather conditions, the Moon's phase and the telescope's angle (how much atmosphere it looks through). These will influence how many photons fall onto the pixels within a CCD and, consequently, the flux of the star.</p>
<p>Comparing the flux of the target star with the flux of the reference stars highlights how these effects simultaneously affect all the stars. The planet's transit in the target star is barely visible as a depression in the flux compared to the reference stars. We will deal with all these effects by performing <strong>differential photometry</strong>.</p>
<p>Comparison stars appear in the same field of view as your target object and, therefore, will be affected similarly by the factors discussed. So, by measuring comparison stars and your target object in each image, any variations of the comparison stars can be attributed to environmental factors/seeing effects and can be used to correct the measurements of your target object. In other words, the comparison stars are measured and used as a reference (hence <em>reference stars</em>) to cancel out the instrumental effects in the measurements of the target star.</p>
<p>If we assume that these effects (seeing variation, weather conditions, and an increase in background luminosity) affect the observed fluxes proportionally, e.g., an increase in cloud coverage causes a decrease in the flux of a factor that is the same for all the stars. We can remove these effects by dividing the flux of the target star by the flux of the reference star. To increase the precision of our correction, we can also divide the flux tìof the target star by <em>the sum</em> of the fluxes of the reference stars, including only those reference stars that do not show anomalous behaviour (e.g., strange variation in the absolute flux or sky background not observed in other stars)</p>


In [ ]:


plt.figure(figsize=(8,4))
plt.scatter(target_ap08.bjd_tdb-time_offset, target_ap08.aperture/reference01_ap08.aperture, s=2, c='C0', label='Ref #1')
plt.xlabel('BJD-TDB - {0:.1f} [days]'.format(time_offset))
plt.ylabel('Differential photometry')
plt.ylim(0.180, 0.190)
plt.legend()
plt.show()


plt.figure(figsize=(8,4))
plt.scatter(target_ap08.bjd_tdb-time_offset, target_ap08.aperture/reference02_ap08.aperture, s=2, c='C1', label='Ref #2')
plt.xlabel('BJD-TDB - {0:.1f} [days]'.format(time_offset))
plt.ylabel('Differential photometry')
plt.ylim(0.88, 0.96)
plt.legend()
plt.show()

plt.figure(figsize=(8,4))
plt.scatter(target_ap08.bjd_tdb-time_offset, target_ap08.aperture/(reference01_ap08.aperture+reference02_ap08.aperture), s=2, c='C3', label='Sum of refs')
plt.xlabel('BJD-TDB - {0:.1f} [days]'.format(time_offset))
plt.ylabel('Differential photometry')
plt.ylim(0.15, 0.16)
plt.legend()
plt.show()





<h2 id="Selection-of-the-best-differential-photometry">Selection of the best differential photometry</h2><p>The plot above shows the three differential photometry using the two reference stars and the sum of the flux of the two reference stars selected in this example. With the increasing flux of the reference star, the differential photometry will assume smaller values, so the plot as it is now is not highly informative, as it gives the deceiving idea that the differential photometry obtained with the fainter star (the second reference star) is the best one in highlighting the transit of the planet.</p>
<p>We need to find a quantitative criterion to select the <em>best</em> differential photometry. How do we define the <em>best</em> differential photometry? Generally speaking, the answer relies on the physical phenomenon we want to study. In our specific case, we are interested in modelling the transit of the planet, and we know that in the short time around the observations, the flux of the star can be considered constant, so the measurements obtained before and after the transit provide an excellent opportunity to identify the differential photometry approach delivering the best results. As we expect a constant value of the differential photometry outside the transit, the standard deviation of this portion of the light curve can provide a quantitative criterion for selecting the best photometric extraction.</p>
<p>First, we <strong>manually select the exclusion interval for the transit</strong>. We do not need to pinpoint precisely the beginning and end of the transit; we can be more relaxed and exclude some observations before and after the transit. What is really important is always to use the same selection (the same epochs) to
compute the standard deviation across different combinations of reference stars and aperture selections.</p>
<p>For my target, I chose:</p>
<div class="highlight"><pre><span></span><span class="n">ingress_time</span> <span class="o">=</span> <span class="mf">2459667.320</span>
<span class="n">egress_time</span> <span class="o">=</span> <span class="mf">2459667.41</span>
</pre></div>


In [ ]:


ingress_time = 2459667.32
egress_time = 2459667.41

differential_ap08_ref01 = target_ap08.aperture/reference01_ap08.aperture
differential_ap08_ref02 = target_ap08.aperture/reference02_ap08.aperture
differential_ap08_allref = target_ap08.aperture/(reference01_ap08.aperture + reference02_ap08.aperture)

differential_ap08_ref01_error = 0.001 #### THESE ARE RANDOM VALUES! You must compute the error by yourself
differential_ap08_ref02_error = 0.001 #### THESE ARE RANDOM VALUES! You must compute the error by yourself
differential_ap08_allref_error = 0.001 #### THESE ARE RANDOM VALUES! You must compute the error by yourself


plt.figure()
plt.scatter(target_ap08.bjd_tdb-time_offset, differential_ap08_allref, s=2, label='All refs')
plt.errorbar(target_ap08.bjd_tdb-time_offset, differential_ap08_allref, yerr=differential_ap08_allref_error, fmt=' ', c='k', alpha=0.25, zorder=-1) 
plt.legend()
plt.axvline(ingress_time-time_offset, c='C3')
plt.axvline(egress_time-time_offset, c='C3')
plt.xlabel('BJD-TDB - {0:.1f} [days]'.format(time_offset))
plt.ylabel('Differential photometry')
plt.ylim(0.15, 0.159)
plt.show()





<p>I use the <code>ingress_time</code> and <code>egress_time</code> to select the points outside the transit. <br/>
I also included a criterion to exclude the points with <code>sky_background</code> too high, as the observations started right at the beginning of the night.</p>


In [ ]:


bjd_tdb = target_ap08.bjd_tdb
bjd_median = np.median(target_ap08.bjd_tdb)


# https://numpy.org/doc/stable/reference/routines.polynomials.package.html#module-numpy.polynomial
from numpy.polynomial import Polynomial

out_transit_selection = (target_ap08.bjd_tdb < ingress_time)  | ( target_ap08.bjd_tdb> egress_time)
# Below an example if you need to exclude some additional features
out_transit_selection = ((target_ap08.bjd_tdb < ingress_time)  | ( target_ap08.bjd_tdb> egress_time)) & (target_ap08.sky_background< 1000)





<p>However, a direct comparison of the standard deviation of the differential photometry computed outside the transit is not possible, as the absolute value depends on the ratio between the brightness of the target and reference star(s).</p>
<p>To better understand this point, let's do an example. Imagine two cases where the average values of the differential photometry are $df_1 = 0.8 \pm 0.2$ and $df_2 = 2.0 \pm 0.4 $, respectively. The absolute value of the standard deviation is smaller in the first case. However, the corresponding relative errors are $\epsilon_{df_1} = \sigma_{df_1} / df_1\ = 0.25$, $\epsilon_{df_2} = 0.20 $, i.e., the relative error of the second differential photometry is smaller. To properly compare the two differential photometry, we must first normalise each of them by their average values so that the <code>numpy.std</code> function will provide an estimate of the <strong>relative error</strong> associated with the light curve.</p>
<p>I use a polynomial trend in the following example to normalize the light curve. A linear or quadratic polynomial may be sufficient if eye inspection reveals the presence of a trend. A trend may appear if the target and reference stars have different spectral types, as the atmospheric extinction strongly depends on the wavelength.</p>
<p>Finally, the standard deviation is computed on the normalized light curve.</p>


In [ ]:


order_polyfit = 2

poly_ap08_ref01_deg02_pfit = Polynomial.fit(bjd_tdb[out_transit_selection]-bjd_median, differential_ap08_ref01[out_transit_selection], deg=order_polyfit)
poly_ap08_ref02_deg02_pfit = Polynomial.fit(bjd_tdb[out_transit_selection]-bjd_median, differential_ap08_ref02[out_transit_selection], deg=order_polyfit)
poly_ap08_allref_deg02_pfit = Polynomial.fit(bjd_tdb[out_transit_selection]-bjd_median, differential_ap08_allref[out_transit_selection], deg=order_polyfit)

differential_ap08_ref01_normalized = differential_ap08_ref01 / poly_ap08_ref01_deg02_pfit(bjd_tdb-bjd_median)
differential_ap08_ref02_normalized = differential_ap08_ref02 / poly_ap08_ref02_deg02_pfit(bjd_tdb-bjd_median)
differential_ap08_allref_normalized = differential_ap08_allref / poly_ap08_allref_deg02_pfit(bjd_tdb-bjd_median)

differential_ap08_ref01_normalized_error = 0.001 #### THESE ARE RANDOM VALUES! You must compute the error by yourself
differential_ap08_ref02_normalized_error = 0.001 #### THESE ARE RANDOM VALUES! You must compute the error by yourself
differential_ap08_allref_normalized_error = 0.001 #### THESE ARE RANDOM VALUES! You must compute the error by yourself

print('Standard deviation aperture 08 reference #1:    {0:.7f}'.format(np.std(differential_ap08_ref01_normalized[out_transit_selection])))
print('Standard deviation aperture 08 reference #2:    {0:.7f}'.format(np.std(differential_ap08_ref02_normalized[out_transit_selection])))
print('Standard deviation aperture 08 all references : {0:.7f}'.format(np.std(differential_ap08_allref_normalized[out_transit_selection])))

plt.figure()

plt.scatter(bjd_tdb-time_offset, differential_ap08_ref01 / np.median(differential_ap08_ref01), s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap08_ref02/ np.median(differential_ap08_ref02), s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap08_allref/ np.median(differential_ap08_allref), s=2)

plt.plot(bjd_tdb-time_offset, poly_ap08_ref01_deg02_pfit(bjd_tdb-bjd_median)/ np.median(differential_ap08_ref01))
plt.plot(bjd_tdb-time_offset, poly_ap08_ref02_deg02_pfit(bjd_tdb-bjd_median)/ np.median(differential_ap08_ref02))
plt.plot(bjd_tdb-time_offset, poly_ap08_allref_deg02_pfit(bjd_tdb-bjd_median)/ np.median(differential_ap08_allref))

plt.axvline(ingress_time-time_offset, c='C3')
plt.axvline(egress_time-time_offset, c='C3')
plt.ylim(0.975, 1.025)

plt.xlabel('BJD-TDB - {0:.1f} [days]'.format(time_offset))
plt.ylabel('Rescaled differential photometry')

plt.show()
plt.figure()
plt.scatter(bjd_tdb-time_offset, differential_ap08_ref01_normalized, s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap08_ref02_normalized, s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap08_allref_normalized, s=2)

plt.axvline(ingress_time-time_offset, c='C3')
plt.axvline(egress_time-time_offset, c='C3')
plt.ylim(0.975, 1.025)

plt.xlabel('BJD-TDB - {0:.1f} [days]'.format(time_offset))
plt.ylabel('Normalized differential photometry')
plt.show()





<p>From the results above, we conclude that the <strong>first reference star</strong> is providing the best differential photometry. <br/>
In general, you would expect that increasing the number of reference stars will cause a decrease in the scatter of the out-of-transit part as the signal-to-noise ratio of the denominator of the differential photometry increases. If it is not the case, most likely, there is some problem with the reference star (intrinsic variability, improper sky subtraction, ...).<br/>
However, it is essential to notice that the standard deviation may decrease even if <strong>the additional reference star is not delivering a reliable differential photometry</strong>, as you can see in this case: for example, in the differential photometry with the second reference target, the out-of-transit has a different overall curvature, and the planetary transit is showing a bump that is absent when using the first reference star, mostly caused by a variation of the second reference.</p>



<p>From the results above, we conclude that the <strong>first reference star</strong> is providing the best differential photometry.
In general, you would expect that increasing the number of reference stars will cause a decrease in the scatter of the out-of-transit part as the signal-to-noise ratio of the denominator of the differential photometry increases. If it is not the case, most likely, there is some problem with the reference star (intrinsic variability, improper sky subtraction, ...).</p>



<h2 id="Using-different-photometric-parameters">Using different photometric parameters</h2><p>Using the <code>AperturePhotometry</code> class is highly convenient for exploring the effects of different photometric parameters. Below, I report an example using the same reference stars but with a different aperture radius.</p>


In [ ]:


from time import time
t0 = time()

aperture = 5
inner_radius = 18
outer_radius = 23

target_ap05 = AperturePhotometry()
target_ap05.provide_aperture_parameters(inner_radius, outer_radius, aperture, x_target, y_target)
target_ap05.aperture_photometry()

reference01_ap05 = AperturePhotometry()
reference01_ap05.provide_aperture_parameters(inner_radius, outer_radius, aperture, x_reference_01, y_reference_01)
reference01_ap05.aperture_photometry()

reference02_ap05 = AperturePhotometry()
reference02_ap05.provide_aperture_parameters(inner_radius, outer_radius, aperture, x_reference_02, y_reference_02)
reference02_ap05.aperture_photometry()

t1 = time()
print('elapsed_time=', t1-t0)




In [ ]:


differential_ap05_ref01 = target_ap05.aperture/reference01_ap05.aperture
differential_ap05_ref02 = target_ap05.aperture/reference02_ap05.aperture
differential_ap05_allref = target_ap05.aperture/(reference01_ap05.aperture + reference02_ap05.aperture)

order_polyfit = 2

# https://numpy.org/doc/stable/reference/routines.polynomials.package.html#module-numpy.polynomial
from numpy.polynomial import Polynomial
#out_transit_selection = (bjd_tdb < 2459318.46509)  | ( bjd_tdb> 2459318.6100)
poly_ap05_ref01_deg02_pfit = Polynomial.fit(bjd_tdb[out_transit_selection]-bjd_median, differential_ap05_ref01[out_transit_selection], deg=order_polyfit)
poly_ap05_ref02_deg02_pfit = Polynomial.fit(bjd_tdb[out_transit_selection]-bjd_median, differential_ap05_ref02[out_transit_selection], deg=order_polyfit)
poly_ap05_allref_deg02_pfit = Polynomial.fit(bjd_tdb[out_transit_selection]-bjd_median, differential_ap05_allref[out_transit_selection], deg=order_polyfit)

differential_ap05_ref01_normalized = differential_ap05_ref01 / poly_ap05_ref01_deg02_pfit(bjd_tdb-bjd_median)
differential_ap05_ref02_normalized = differential_ap05_ref02 / poly_ap05_ref02_deg02_pfit(bjd_tdb-bjd_median)
differential_ap05_allref_normalized = differential_ap05_allref / poly_ap05_allref_deg02_pfit(bjd_tdb-bjd_median)


plt.figure()

plt.scatter(bjd_tdb-time_offset, differential_ap05_ref01 / np.median(differential_ap05_ref01), s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap05_ref02/ np.median(differential_ap05_ref02), s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap05_allref/ np.median(differential_ap05_allref), s=2)

plt.plot(bjd_tdb-time_offset, poly_ap05_ref01_deg02_pfit(bjd_tdb-bjd_median)/ np.median(differential_ap05_ref01))
plt.plot(bjd_tdb-time_offset, poly_ap05_ref02_deg02_pfit(bjd_tdb-bjd_median)/ np.median(differential_ap05_ref02))
plt.plot(bjd_tdb-time_offset, poly_ap05_allref_deg02_pfit(bjd_tdb-bjd_median)/ np.median(differential_ap05_allref))

plt.axvline(ingress_time-time_offset, c='C3')
plt.axvline(egress_time-time_offset, c='C3')
plt.ylim(0.975, 1.025)

plt.xlabel('BJD-TDB - {0:.1f} [days]'.format(time_offset))
plt.ylabel('Rescaled differential photometry')
plt.show()



plt.figure()
plt.scatter(bjd_tdb-time_offset, differential_ap05_ref01_normalized, s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap05_ref02_normalized, s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap05_allref_normalized, s=2)

plt.axvline(ingress_time-time_offset, c='C3')
plt.axvline(egress_time-time_offset, c='C3')
plt.ylim(0.95, 1.05)
plt.xlabel('BJD-TDB - {0:.1f} [days]'.format(time_offset))
plt.ylabel('Normalized differential photometry')
plt.show()



print('Standard deviation aperture 05 reference #1:    {0:.7f}'.format(np.std(differential_ap05_ref01_normalized[out_transit_selection])))
print('Standard deviation aperture 05 reference #2:    {0:.7f}'.format(np.std(differential_ap05_ref02_normalized[out_transit_selection])))
print('Standard deviation aperture 05 all references : {0:.7f}'.format(np.std(differential_ap05_allref_normalized[out_transit_selection])))




In [ ]:


from time import time
t0 = time()

aperture = 10
inner_radius = 18
outer_radius = 23

target_ap10 = AperturePhotometry()
target_ap10.provide_aperture_parameters(inner_radius, outer_radius, aperture, x_target, y_target)
target_ap10.aperture_photometry()

reference01_ap10 = AperturePhotometry()
reference01_ap10.provide_aperture_parameters(inner_radius, outer_radius, aperture, x_reference_01, y_reference_01)
reference01_ap10.aperture_photometry()

reference02_ap10 = AperturePhotometry()
reference02_ap10.provide_aperture_parameters(inner_radius, outer_radius, aperture, x_reference_02, y_reference_02)
reference02_ap10.aperture_photometry()

t1 = time()
print('elapsed_time=', t1-t0)

differential_ap10_ref01 = target_ap10.aperture/reference01_ap10.aperture
differential_ap10_ref02 = target_ap10.aperture/reference02_ap10.aperture
differential_ap10_allref = target_ap10.aperture/(reference01_ap10.aperture + reference02_ap10.aperture)

order_polyfit = 2

# https://numpy.org/doc/stable/reference/routines.polynomials.package.html#module-numpy.polynomial
from numpy.polynomial import Polynomial
#out_transit_selection = (bjd_tdb < 2459318.46509)  | ( bjd_tdb> 2459318.6100)
poly_ap10_ref01_deg02_pfit = Polynomial.fit(bjd_tdb[out_transit_selection]-bjd_median, differential_ap10_ref01[out_transit_selection], deg=order_polyfit)
poly_ap10_ref02_deg02_pfit = Polynomial.fit(bjd_tdb[out_transit_selection]-bjd_median, differential_ap10_ref02[out_transit_selection], deg=order_polyfit)
poly_ap10_allref_deg02_pfit = Polynomial.fit(bjd_tdb[out_transit_selection]-bjd_median, differential_ap10_allref[out_transit_selection], deg=order_polyfit)

differential_ap10_ref01_normalized = differential_ap10_ref01 / poly_ap10_ref01_deg02_pfit(bjd_tdb-bjd_median)
differential_ap10_ref02_normalized = differential_ap10_ref02 / poly_ap10_ref02_deg02_pfit(bjd_tdb-bjd_median)
differential_ap10_allref_normalized = differential_ap10_allref / poly_ap10_allref_deg02_pfit(bjd_tdb-bjd_median)


plt.figure()

plt.scatter(bjd_tdb-time_offset, differential_ap10_ref01 / np.median(differential_ap10_ref01), s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap10_ref02/ np.median(differential_ap10_ref02), s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap10_allref/ np.median(differential_ap10_allref), s=2)

plt.plot(bjd_tdb-time_offset, poly_ap10_ref01_deg02_pfit(bjd_tdb-bjd_median)/ np.median(differential_ap10_ref01))
plt.plot(bjd_tdb-time_offset, poly_ap10_ref02_deg02_pfit(bjd_tdb-bjd_median)/ np.median(differential_ap10_ref02))
plt.plot(bjd_tdb-time_offset, poly_ap10_allref_deg02_pfit(bjd_tdb-bjd_median)/ np.median(differential_ap10_allref))

plt.axvline(ingress_time-time_offset, c='C3')
plt.axvline(egress_time-time_offset, c='C3')
plt.ylim(0.975, 1.025)

plt.xlabel('BJD-TDB - {0:.1f} [days]'.format(time_offset))
plt.ylabel('Rescaled differential photometry')
plt.show()



plt.figure()
plt.scatter(bjd_tdb-time_offset, differential_ap10_ref01_normalized, s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap10_ref02_normalized, s=2)
plt.scatter(bjd_tdb-time_offset, differential_ap10_allref_normalized, s=2)

plt.axvline(ingress_time-time_offset, c='C3')
plt.axvline(egress_time-time_offset, c='C3')
plt.ylim(0.95, 1.05)
plt.xlabel('BJD-TDB - {0:.1f} [days]'.format(time_offset))
plt.ylabel('Normalized differential photometry')
plt.show()



print('Standard deviation aperture 10 reference #1:    {0:.7f}'.format(np.std(differential_ap10_ref01_normalized[out_transit_selection])))
print('Standard deviation aperture 10 reference #2:    {0:.7f}'.format(np.std(differential_ap10_ref02_normalized[out_transit_selection])))
print('Standard deviation aperture 10 all references : {0:.7f}'.format(np.std(differential_ap10_allref_normalized[out_transit_selection])))





<p>By comparing the standard deviation of the normalised differential photometries, in my case, an aperture of 5 pixels produces the worst results, while an aperture of 8 pixels seems to be the optimal choice compared to an aperture of 10 pixels. In all cases, the best results are obtained by using the reference number 1. The inclusion of the second reference improve the results only marginally. Depending on the weather conditions and the peculiarities of your target and reference stars, you may get better results with another combination of aperture and reference stars. Note that the brightest star may not necessarily be the best reference if, for example, the reference star is in a binary system with the companion partially entering and exiting the aperture radius or the sky annulus with varying seeing.</p>
<p>Finally, we save the differential light curve and the associated values into a file for later use.</p>
<p><strong>HOMEWORK</strong>: do not forget to compute the errors associated with the differential photometry.</p>


In [ ]:


pickle.dump(bjd_tdb, open('taste_bjdtdb.p','wb'))
pickle.dump(differential_ap08_ref01_normalized, open('differential_ap08_ref01_normalized.p','wb'))
pickle.dump(differential_ap08_ref01_normalized_error, open('differential_ap08_ref01_normalized_error.p','wb'))
pickle.dump(differential_ap08_allref_normalized, open('differential_ap08_allref_normalized.p','wb'))
pickle.dump(differential_ap08_allref_normalized_error, open('differential_ap08_allref_normalized_error.p','wb'))

pickle.dump(differential_ap08_ref01, open('differential_ap08_ref01.p','wb'))
pickle.dump(differential_ap08_ref01_error, open('differential_ap08_ref01_error.p','wb'))

pickle.dump(differential_ap08_allref, open('differential_ap08_allref.p','wb'))
pickle.dump(differential_ap08_allref_error, open('differential_ap08_allref_error.p','wb'))


